# Multivariate distributions

Static Gaussian and Student-t copulas are used as benchmarks. Two dynamic
models are fitted with SCAR-TM-OU using the default optimizer settings:

- `EquicorrGaussianCopula`: time-varying common correlation;
- `StochasticStudentCopula`: time-varying degrees of freedom and a constant
  correlation matrix (the default `corr_mode="fixed"`).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from pyscarcopula._utils import pobs
from pyscarcopula import (
    EquicorrGaussianCopula,
    GaussianCopula,
    StochasticStudentCopula,
    StudentCopula,
)
from pyscarcopula.stattests import gof_test

## Data

In [2]:
TICKERS = ["BTC-USD", "ETH-USD", "BNB-USD", "ADA-USD"]
ROWS = slice(0, 400)

DATA_DIR = Path("data") if Path("data").exists() else Path("..") / "data"
prices = pd.read_csv(DATA_DIR / "crypto_prices.csv", index_col=0, sep=";")
returns = np.log(prices[TICKERS] / prices[TICKERS].shift(1)).dropna().iloc[ROWS]
u = pobs(returns.to_numpy())
d = u.shape[1]

print(f"T={len(u)}, d={d}")

T=400, d=4


## Fit models

In [3]:
gaussian = GaussianCopula()
gaussian_result = gaussian.fit(u)

student = StudentCopula()
student_result = student.fit(u)

equicorr = EquicorrGaussianCopula(d=d)
equicorr_result = equicorr.fit(u, method="scar-tm-ou")

# Default mode: the correlation matrix is estimated once and then fixed.
stochastic_student = StochasticStudentCopula(d=d)
stochastic_student_result = stochastic_student.fit(
    u, method="scar-tm-ou"
)

## Main fit results

In [4]:
results = {
    "Gaussian": gaussian_result,
    "Student-t": student_result,
    "Equicorr Gaussian SCAR": equicorr_result,
    "Stochastic Student SCAR": stochastic_student_result,
}
models = {
    "Gaussian": gaussian,
    "Student-t": student,
    "Equicorr Gaussian SCAR": equicorr,
    "Stochastic Student SCAR": stochastic_student,
}

fit_table = pd.DataFrame({
    name: {
        "method": result.method,
        "success": result.success,
        "log_likelihood": result.log_likelihood,
        "n_params": result.n_params,
    }
    for name, result in results.items()
}).T
fit_table["gof_pvalue"] = {
    name: gof_test(model, u, to_pobs=False).pvalue
    for name, model in models.items()
}
fit_table

,method,success,log_likelihood,n_params,gof_pvalue
Gaussian,MLE,True,580.84234,6,8.459345e-08
Student-t,MLE,True,655.381794,7,2.374560e-01
Equicorr Gaussian SCAR,SCAR-TM-OU,True,678.766407,3,1.905019e-01
Stochastic Student SCAR,SCAR-TM-OU,True,662.527779,3,2.690330e-01


## Dynamic parameters

In [5]:
rho_t = equicorr.predictive_mean(u)
df_t = stochastic_student.predictive_mean(u)

pd.DataFrame({
    "equicorrelation": [rho_t.mean(), rho_t.min(), rho_t.max()],
    "degrees_of_freedom": [df_t.mean(), df_t.min(), df_t.max()],
}, index=["mean", "min", "max"])

,equicorrelation,degrees_of_freedom
mean,0.734203,6.868135
min,0.382195,2.919001
max,0.928391,13.290032
